# Base Batch-Hartigan Convergence Re-run (v2)

Runs the low-dimensional Batch-Hartigan failure-mode study (paper Section 5.1) end-to-end in four stages.
**Run the cells top to bottom.** Stages 1 and 3 are the local verification path (seconds to a minute).

1. **Unit tests** — verify loop detection, loop-size correctness, ground-truth data generation, and that the vectorized pass matches the reference loop exactly.
2. **System test** — measure this machine (CPUs, RAM) and estimate the full run's wall-clock + memory.
3. **Smoke test** — a tiny end-to-end sweep that exercises the whole pipeline (writes `results_smoke.csv`).
4. **Full run** — the real 372,400-regime sweep (writes `../results/data/results_v2.csv`; the shipped copy is gzipped). Guarded by a flag; run only after 1–3 look good.

### Design notes
- **Ground-truth data generation** (real cluster structure); labels are discarded for the failure-mode study.
- `intracluster_variance` (intra) and `intercluster_variance` (inter) are each swept `1–10`.
- **Vectorized** core pass (`compare_batch_hart`), bit-equivalent to the reference loop (`compare_batch_hart_reference`, checked in stage 1) but ~10–20× faster.
- **Loop tracking**: per-regime `mean_loop_size` / `max_loop_size`, `mean_iters_before_loop` / `max_iters_before_loop`, a `loop_size_histogram`, and a `max_iter_count` (trials that hit the 500-iteration budget).
- On a real cycle the algorithm **stops at detection** — looping trials record their (small) detection iteration instead of spinning to `max_iter`.

> The simulation uses only `numpy` (no `scikit-learn` for the base run). Multiprocessing workers live in `run_simulation.py`, so they stay picklable under both `fork` (Linux) and `spawn` (macOS) start methods.

In [ ]:
import multiprocessing
import platform
import sys
import numpy as np

# default_rng requires numpy >= 1.17; fail fast with a clear message on old envs.
assert tuple(int(x) for x in np.__version__.split(".")[:2]) >= (1, 17), \
    f"numpy >= 1.17 required, got {np.__version__}"

from batch_hartigan import batch_hartigan as run_bh
from batch_hartigan import generate_ground_truth_data, loop_checker
import run_simulation as rs
# NOTE: if you edit batch_hartigan.py / run_simulation.py, restart the kernel
# (Kernel > Restart) and re-run from the top so the pool workers pick up the new code.

print("python", sys.version.split()[0], "| numpy", np.__version__)
print("default grid:",
      f"{len(rs.AMOUNT_OF_CLUSTERS_VALUES)} x {len(rs.DIM_VALUES)} x "
      f"{len(rs.INTRACLUSTER_VARIANCE_VALUES)} x {len(rs.CLUSTER_SIZE_VALUES)} x "
      f"{len(rs.INTERCLUSTER_VARIANCE_VALUES)} = {len(rs.build_regimes()):,} regimes")

## 1. Unit tests

Deterministic checks (fixed seeds) of the pieces that matter for the base study:
loop **detection** + loop **size**, the clean-convergence guard, and the ground-truth generator.
These run in-process and finish in a couple of seconds.

In [ ]:
def _feed(seq):
    """Feed assignment vectors through loop_checker; return (loop, loop_size) per step."""
    memory, out = {}, []
    for counter, a in enumerate(seq):
        loop, size, memory = loop_checker(np.array(a, dtype=np.int64), memory, counter)
        out.append((loop, size))
    return out

# --- loop_checker: detection + size ---------------------------------------
assert _feed([[0, 1, 0], [1, 0, 1], [0, 0, 1]]) == [(False, 0)] * 3          # 3 unique -> never
assert _feed([[0, 1, 0], [1, 0, 1], [0, 1, 0]])[2] == (True, 2)              # A,B,A   -> size 2
assert _feed([[0, 0, 1], [0, 1, 0], [1, 0, 0], [0, 0, 1]])[3] == (True, 3)   # A,B,C,A -> size 3
assert _feed([[0, 0], [0, 1], [1, 1], [1, 0], [0, 1]])[4] == (True, 3)       # offset  -> size 3
assert (hash(np.array([0, 1, 0], dtype=np.int64).tobytes())
        == hash(np.array([0, 1, 0], dtype=np.int64).tobytes()))             # tobytes hashing
print("[ok] loop_checker: no-loop, size-2, size-3, offset-3, hash-equality")

# --- generate_ground_truth_data: shapes, labels, reproducibility ----------
rng = np.random.default_rng(0)
cents, data, init, labels = generate_ground_truth_data(rng, 20, 3, 2, 0.3, 25.0)
assert data.shape == (60, 2) and labels.shape == (60,)
assert (np.bincount(labels) == 20).all()          # each cluster has cluster_size points
assert init.shape == (3, 2) and cents.shape == (3, 2)
c2, d2, _, l2 = generate_ground_truth_data(np.random.default_rng(0), 20, 3, 2, 0.3, 25.0)
assert np.array_equal(data, d2) and np.array_equal(labels, l2)  # same seed -> same data
print("[ok] generate_ground_truth_data: shapes, label coverage (20 each), reproducibility")

# --- batch_hartigan: termination + clean-convergence guard ----------------
n_clean = 0
for s in range(30):
    rng = np.random.default_rng(s)
    cents, data, init, _ = generate_ground_truth_data(rng, 25, 3, 2, 0.3, 30.0)  # well separated
    assigns, cents, counter, loop, size, zero = run_bh(data, init, cents, 3, max_iter=500)
    assert counter <= 500
    assert (size >= 2) if loop else (size == 0)    # a converged fixed point is NOT a size-1 loop
    n_clean += (not loop and not zero)
assert n_clean >= 20
print(f"[ok] batch_hartigan: termination + clean-convergence guard ({n_clean}/30 clean)")

# --- batch_hartigan: real loops are detected, with size >= 2 and early stop
def _mk(seed):
    rng = np.random.default_rng(seed)
    cents, data, init, _ = generate_ground_truth_data(rng, 3, 3, 2, 10.0, 2.0)  # high overlap
    return data, init, cents

found, n_loops = None, 0
for s in range(600):
    assigns, cents, counter, loop, size, zero = run_bh(*_mk(s), 3, max_iter=500)
    if loop:
        n_loops += 1
        assert size >= 2 and counter < 500        # genuine cycle, stopped at detection
        found = found or (s, size, counter)
assert found is not None, "expected to find at least one looping seed"
print(f"[ok] batch_hartigan: detected real loops ({n_loops}/600 seeds); "
      f"first @seed={found[0]} size={found[1]} counter={found[2]}")

# --- determinism: same seed -> identical outcome --------------------------
a1, c1, cnt1, lp1, sz1, z1 = run_bh(*_mk(found[0]), 3, max_iter=500)
a2, c2, cnt2, lp2, sz2, z2 = run_bh(*_mk(found[0]), 3, max_iter=500)
assert cnt1 == cnt2 and lp1 == lp2 and sz1 == sz2 and np.array_equal(a1, a2)
print(f"[ok] determinism: seed={found[0]} reproduces counter/loop/size/assigns")

print("\nALL UNIT TESTS PASSED")

In [ ]:
# --- vectorized compare_batch_hart == reference loop (bit-equivalent) ------
from batch_hartigan import compare_batch_hart, compare_batch_hart_reference, compare_init
import batch_hartigan as _bh

def _rand_cfg(rng):
    m = int(rng.integers(2, 16)); dim = int(rng.integers(1, 16)); cs = int(rng.integers(2, 40))
    data = rng.normal(size=(cs * m, dim))
    cents = data[rng.choice(cs * m, size=m, replace=False)].copy()
    mode = rng.integers(0, 4)
    if   mode == 0: a = compare_init(data, cents)                              # nearest-centroid
    elif mode == 1: a = rng.integers(0, m, size=cs * m)                        # random
    elif mode == 2: a = rng.integers(0, max(1, m - int(rng.integers(1, m))), size=cs * m)  # empties
    else:           a = np.zeros(cs * m, dtype=int); a[0] = 1 % m              # singleton (pop==1)
    return data, cents, np.asarray(a, dtype=np.int64), m

rng = np.random.default_rng(0); bad = 0
for _ in range(3000):
    data, cents, a, m = _rand_cfg(rng)
    empty = (np.bincount(a, minlength=m) == 0).any()
    cr = cents.copy(); ar, zr = compare_batch_hart_reference(data, cr, a.copy(), m, False)
    cv = cents.copy(); av, zv = compare_batch_hart(data, cv, a.copy(), m, False)
    if empty:
        bad += not (zr and zv)                                  # both flag zero; assign discarded
    else:
        bad += not (np.array_equal(ar, av) and zr == zv
                    and np.array_equal(cr, cv, equal_nan=True))  # NaN from newly-empty cluster ok
assert bad == 0, f"{bad} pass-level mismatches"
print("[ok] vectorized compare_batch_hart == reference over 3000 random configs")

def _run(seed, regime, fn):
    _bh.compare_batch_hart = fn
    cs, m, dim, iv, ev = regime
    r = np.random.default_rng(seed)
    ce, da, ini, _l = generate_ground_truth_data(r, cs, m, dim, iv, ev)
    return _bh.batch_hartigan(da, ini, ce, m, max_iter=500)

mism = 0
for regime in [(3, 3, 2, 10, 2), (3, 5, 2, 10, 2), (10, 5, 3, 6, 6), (2, 3, 2, 8, 2)]:
    for seed in range(300):
        a1, c1, cnt1, lp1, sz1, z1 = _run(seed, regime, compare_batch_hart_reference)
        a2, c2, cnt2, lp2, sz2, z2 = _run(seed, regime, compare_batch_hart)
        ok = (cnt1 == cnt2 and z1 == z2 and lp1 == lp2 and sz1 == sz2)
        if not z1:
            ok = ok and np.array_equal(a1, a2)
        mism += not ok
_bh.compare_batch_hart = compare_batch_hart   # restore
assert mism == 0, f"{mism} pipeline mismatches"
print("[ok] identical counter/loop/loop_size/zero over 1200 pipeline runs")

## 2. System test (cluster sizing)

Measures this machine and estimates the full run. The estimate times a **uniform random
sample** of regimes in-process (so mean per-trial time is an unbiased estimate of the
grid-average) and extrapolates to the full grid. The parallel figure assumes ~linear
scaling across workers, so treat it as a best case.

This cell itself takes ~10–60 s depending on the machine.

In [ ]:
import os

print("=== Machine ===")
print("platform :", platform.platform())
print("python   :", sys.version.split()[0])
print("numpy    :", np.__version__)
n_cpu = multiprocessing.cpu_count()
print("CPUs     :", n_cpu)
try:
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    print("RAM (GB) :", f"{ram_gb:.1f}")
except (ValueError, AttributeError, OSError):
    print("RAM (GB) : unknown")

n_workers = max(1, n_cpu - 1)
print(f"\nPlanned workers for the full run: {n_workers}")

print("\n=== Runtime estimate (uniform-sample micro-benchmark) ===")
est = rs.estimate_full_run(sample_size=40, trials_per_sample=20, n_workers=n_workers)
print(f"sampled {est['sample_size']} regimes ({est['trials_per_sample']} trials each) "
      f"in {est['sample_wall_seconds']:.1f}s")
print(f"mean per-trial : {est['mean_per_trial_s']*1000:.2f} ms")
wp = est['worst_params']   # (regime_idx, amount_of_clusters, dim, intracluster_variance, cluster_size, intercluster_variance)
print(f"worst per-trial: {est['worst_per_trial_s']*1000:.2f} ms @ "
      f"amount_of_clusters={wp[1]}, dim={wp[2]}, intracluster_variance={wp[3]}, "
      f"cluster_size={wp[4]}, intercluster_variance={wp[5]}")
print(f"full grid      : {est['total_regimes']:,} regimes x {est['full_trials_per_regime']} "
      f"trials = {est['total_trials']:,} trials")
print(f"est. SERIAL    : {est['est_serial_seconds']/3600:.1f} h")
print(f"est. PARALLEL  : {est['est_parallel_seconds']/3600:.2f} h  "
      f"(@ {est['n_workers']} workers, ~linear-scaling assumption)")

print("\n=== Memory / output footprint ===")
max_n = max(rs.CLUSTER_SIZE_VALUES) * max(rs.AMOUNT_OF_CLUSTERS_VALUES)
max_mb = max_n * max(rs.DIM_VALUES) * 8 / 1e6
print(f"peak data array per worker: {max_n} pts x {max(rs.DIM_VALUES)} dims x 8B "
      f"= {max_mb:.2f} MB (negligible)")
print(f"output CSV: {est['total_regimes']:,} rows (a few MB)")

## 3. Smoke test (tiny end-to-end)

A small sweep that runs the **same code path** as the full run (ground-truth generation →
`batch_hartigan` → loop/size tracking → parallel pool → CSV) at tiny scale. It mixes
well-separated and high-overlap regimes so loops actually occur. Writes `results_smoke.csv`.

In [ ]:
smoke_regimes = rs.build_regimes(
    amount_of_clusters_values=[2, 3],
    dim_values=[2, 5],
    intracluster_variance_values=[2, 10],
    cluster_size_values=[3, 20],
    intercluster_variance_values=[2],
)   # 2 x 2 x 2 x 2 x 1 = 16 regimes

smoke_summary = rs.run_sweep(
    smoke_regimes,
    trials_per_regime=50,
    csv_path="results_smoke.csv",
    n_workers=max(1, multiprocessing.cpu_count() - 1),
    verbose=True,
)

## 4. Full run

The real sweep: **372,400 regimes × 1000 trials → `../results/data/results_v2.csv`**.
Estimated wall-clock is printed by the System Test cell above.

**Guarded** by `RUN_FULL_SWEEP` so a stray *Run All* can't launch a multi-hour job.
Set it to `True` only after stages 1–3 look good, then run this cell.

> **Output location:** `../results/data/results_v2.csv`, next to the shipped, gzipped copy
> `results_v2.csv.gz` (which this never overwrites). Per-trial seeds come from numpy's unseeded
> global RNG (`create_rng` in `batch_hartigan.py`), so a re-run reproduces the regime grid and the
> failure rates up to Monte-Carlo noise, not the exact counts.

In [ ]:
RUN_FULL_SWEEP = False   # <-- set True to launch the full sweep

if RUN_FULL_SWEEP:
    full_summary = rs.run_sweep(csv_path="../results/data/results_v2.csv")   # full grid, 1000 trials/regime
else:
    n_reg = len(rs.build_regimes())
    print("RUN_FULL_SWEEP is False — set it to True to launch the full sweep.")
    print(f"Full grid: {n_reg:,} regimes x {rs.TRIALS_PER_REGIME} trials "
          f"= {n_reg * rs.TRIALS_PER_REGIME:,} trials")